# OOS均值误差与H5期限匹配

整体稳定超额目标继续。本轮依据上一候选缺少独立信息和旧H5每日重配的偏差，登记2个增量、3个版本及同SOXX基准。完整协议 [plan.json](plan.json)，原因见 [diagnosis.md](diagnosis.md)。所有窗口均为复用历史。

In [1]:
from pathlib import Path
import sys, os, json
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
CARD = ROOT / "research" / "oos_uncertainty_horizon_20261009"
PLAN = json.loads((CARD / "plan.json").read_text())
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(ROOT / PLAN["source"]["provider_uri"]), region="us", kernels=1)
calendar = D.calendar(freq="day", future=False)
print("Native calendar", calendar[0], calendar[-1], len(calendar))
print("Registered variants", PLAN["variants"])

[59305:MainThread](2026-10-09 11:38:40,306) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[59305:MainThread](2026-10-09 11:38:40,308) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[59305:MainThread](2026-10-09 11:38:40,308) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


Native calendar 2010-03-11 00:00:00 2026-09-25 00:00:00 4162
Registered variants [{'name': 'hac_daily', 'rebalance_sessions': 1, 'oos_floor': False, 'role': 'matched_canonical_control'}, {'name': 'hac_h5', 'rebalance_sessions': 5, 'oos_floor': False, 'role': 'cadence_candidate'}, {'name': 'oos_floor_h5', 'rebalance_sessions': 5, 'oos_floor': True, 'role': 'uncertainty_candidate'}]


## 时间与估计合同

年度Ridge10只用严格成熟H5标签；模型在各年开始固定，OOF预测不回填。2013–2015用于2016年以前的误差校准。每年误差floor仅用此前成熟OOF残差，b bᵀ＋HAC均值误差加到原训练HAC。没有把实现收益RMSE称为均值置信区间。信号日16:15决策、次日开盘成交；5日HOLD期间允许风险漂移并如实核验。

In [2]:
from src.workflow.oos_uncertainty_horizon import run_oos_uncertainty_horizon
summary, tables, conclusion = run_oos_uncertainty_horizon(
    source_root=ROOT, evidence_root=ROOT, card_dir=CARD,
    recorder_uri=CARD / "qlib_recorder",
)
print({key: summary[key] for key in ["recorder_id", "native_paths", "historical_gates", "objective_achieved"]})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[59305:MainThread](2026-10-09 11:38:41,675) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[59305:MainThread](2026-10-09 11:38:41,676) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[59305:MainThread](2026-10-09 11:38:41,676) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


[59305:MainThread](2026-10-09 11:38:41,693) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[59305:MainThread](2026-10-09 11:39:01,411) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[59305:MainThread](2026-10-09 11:39:01,430) INFO - qlib.workflow - [exp.py:258] - Experiment 351822870072437750 starts running ...


[59305:MainThread](2026-10-09 11:39:01,433) INFO - qlib.workflow - [recorder.py:345] - Recorder 64610b843175452b80e966c5dd99adef starts running under Experiment 351822870072437750 ...


[59305:MainThread](2026-10-09 11:39:05,333) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'recorder_id': '64610b843175452b80e966c5dd99adef', 'native_paths': 8, 'historical_gates': {'hac_h5': False, 'oos_floor_h5': False}, 'objective_achieved': False}


In [3]:
from IPython.display import display, Markdown
display(tables["backtest_benchmark_comparison.csv"][["strategy", "cost_bps", "window", "relative_wealth", "max_drawdown_pct", "cvar5_pct", "turnover"]])
display(tables["gates.csv"])
display(tables["ablation_comparison.csv"][["strategy", "control", "cost_bps", "window", "relative_wealth"]])
display(tables["expert_attribution.csv"])

,strategy,cost_bps,window,relative_wealth,max_drawdown_pct,cvar5_pct,turnover
0,soxx,10.0,development,1.000000,-47.373290,-4.416437,0.999001
1,soxx,10.0,reused_diagnostic,1.000000,-42.789189,-5.295891,0.000000
2,hac_daily,10.0,development,0.979609,-45.294105,-4.346521,78.300046
3,hac_daily,10.0,reused_diagnostic,1.127182,-43.022062,-5.108552,80.446709
4,hac_h5,10.0,development,0.871152,-41.213549,-4.385856,29.111955
5,hac_h5,10.0,reused_diagnostic,0.797533,-42.789189,-5.295891,14.632422
6,oos_floor_h5,10.0,development,1.034313,-41.213549,-4.385799,24.936346
7,oos_floor_h5,10.0,reused_diagnostic,0.893725,-42.789189,-5.295891,9.995005
8,soxx,25.0,development,1.000000,-47.373290,-4.418079,0.997506
9,soxx,25.0,reused_diagnostic,1.000000,-42.789189,-5.295891,0.000000


,cost_bps,aggregate_positive_net_log_excess,minimum_positive_signal_years,median_annual_log_excess_positive,leave_any_one_year_out_log_excess_positive,anchored_max_drawdown_not_worse_than_soxx,cvar5_not_worse_than_soxx,minimum_rolling_positive_excess_rate,stationary_block_20_lower_bound_positive,stationary_block_60_lower_bound_positive,all_pass,positive_years,years,minimum_positive_years,strategy
0,10.0,False,False,False,False,True,True,False,False,False,False,3,7,6,hac_h5
1,25.0,False,False,False,False,True,True,False,False,False,False,2,7,6,hac_h5
2,10.0,True,False,False,False,True,True,False,False,False,False,2,7,6,oos_floor_h5
3,25.0,False,False,False,False,True,True,False,False,False,False,2,7,6,oos_floor_h5


,strategy,control,cost_bps,window,relative_wealth
0,hac_h5,hac_daily,10.0,development,0.889285
1,hac_h5,hac_daily,10.0,reused_diagnostic,0.707546
2,oos_floor_h5,hac_h5,10.0,development,1.187294
3,oos_floor_h5,hac_h5,10.0,reused_diagnostic,1.120612
4,hac_h5,hac_daily,25.0,development,1.057021
5,hac_h5,hac_daily,25.0,reused_diagnostic,0.955674
6,oos_floor_h5,hac_h5,25.0,development,1.115682
7,oos_floor_h5,hac_h5,25.0,reused_diagnostic,1.000000


,window,sessions,start_signal,last_exit,realized_soxl_beta_to_soxx,evidence_scope,cost_bps,candidate,reference,component_unit,...,mean_delta_soxl_weight_pp,minimum_delta_soxl_weight_pp,maximum_delta_soxl_weight_pp,mean_delta_beta,delta_beta_future_soxx_return_correlation,delta_soxl_weight_future_residual_correlation,candidate_turnover,reference_turnover,candidate_sum_fee_rates_bp,reference_sum_fee_rates_bp
0,development,1762,2016-01-04,2023-01-04,2.990504,ex_post_descriptive_not_prediction_or_promotio...,10.0,online_fixed_share,fixed_mixture,summed_daily_simple_return_bp_not_compounded,...,0.158207,-0.377768,0.469553,0.004528,-0.030600,0.034264,1.960878,2.017164,19.608783,20.171638
1,development,1762,2016-01-04,2023-01-04,2.990504,ex_post_descriptive_not_prediction_or_promotio...,25.0,online_fixed_share,fixed_mixture,summed_daily_simple_return_bp_not_compounded,...,0.155734,-0.381952,0.466107,0.004471,-0.030548,0.034330,1.958338,2.015693,48.958451,50.392322
2,reused_diagnostic,934,2023-01-03,2026-09-25,2.974148,ex_post_descriptive_not_prediction_or_promotio...,10.0,online_fixed_share,fixed_mixture,summed_daily_simple_return_bp_not_compounded,...,0.138424,-0.397730,0.841971,0.004104,-0.023343,-0.034604,0.667566,0.691956,6.675664,6.919559
3,reused_diagnostic,934,2023-01-03,2026-09-25,2.974148,ex_post_descriptive_not_prediction_or_promotio...,25.0,online_fixed_share,fixed_mixture,summed_daily_simple_return_bp_not_compounded,...,0.134860,-0.401332,0.836974,0.004024,-0.023339,-0.034643,0.666521,0.691854,16.663026,17.296340


## 中文结论

本节从保存的Qlib结果同步生成。任何失败都保留；本批结束不代表关闭稳定超额目标。

In [4]:
display(Markdown(conclusion))

# OOS均值误差与预测期限实验

运行了统一canonical价格下的H5 Ridge＋训练HAC日调仓对照、H5持有候选、H5持有＋过去年度OOF均值误差惩罚候选。相同预测分别结算10/25bp，全部完整路径由原生Qlib核验。

以下为25bp；2016–2022按信号年归属，后段是2023以后已复用历史，持仓连续：

- development/soxx：收益325.11%，相对SOXX终值1.0000，MaxDD -47.37%，CVaR5 -4.42%，换手1.00。
- reused_diagnostic/soxx：收益398.62%，相对SOXX终值1.0000，MaxDD -42.79%，CVaR5 -5.30%，换手0.00。
- development/hac_daily：收益237.27%，相对SOXX终值0.7934，MaxDD -45.46%，CVaR5 -4.35%，换手49.66。
- reused_diagnostic/hac_daily：收益410.47%，相对SOXX终值1.0238，MaxDD -42.79%，CVaR5 -5.15%，换手47.67。
- development/hac_h5：收益256.50%，相对SOXX终值0.8386，MaxDD -41.39%，CVaR5 -4.34%，换手20.50。
- reused_diagnostic/hac_h5：收益387.84%，相对SOXX终值0.9784，MaxDD -42.79%，CVaR5 -5.30%，换手2.00。
- development/oos_floor_h5：收益297.74%，相对SOXX终值0.9356，MaxDD -41.39%，CVaR5 -4.37%，换手13.66。
- reused_diagnostic/oos_floor_h5：收益387.84%，相对SOXX终值0.9784，MaxDD -42.79%，CVaR5 -5.30%，换手2.00。

候选历史门：{'hac_h5': False, 'oos_floor_h5': False}。整体稳定超额目标仍在进行，不晋升。

OOF误差项与训练内HAC分开登记；它是平均偏差与其估计误差的保守尺度，不保证条件覆盖。期限对照只改变求解频率；OOF版本与相同5日HOLD对照隔离误差惩罚。

调仓时估计风险不高于SOXX；中间HOLD允许持仓漂移，不能声称风险预算逐日保证。年度、危机与滚动窗口结果分别见小表，实际MaxDD/CVaR门未放宽。

旧专家归因显示其主要共用半导体beta轴，不能靠扩大在线学习率补出独立信息。本轮完整保留失败；若信息与费用/期限修正仍不足，下一阶段将独立检验有完整长历史且发布时点合格的外部信息家族。

全部历史已被复用；价格口径、16条次来源行情、开盘成交与现金零收益假设仍有限制。下一轮先依据失败归因登记新的机制，不因本轮结束而关闭用户目标。

Recorder：64610b843175452b80e966c5dd99adef；as_of=2026-09-25。
